# Summarization Evaluator

Evaluates **summarization quality** using ROUGE metrics computed without external dependencies.

For each `(document, reference_summary)` pair the evaluator:
- Prompts the model to summarize the document
- Scores the generated summary against the reference using ROUGE-1, ROUGE-2, and ROUGE-L
- Reports compression ratio (summary words / document words)

**ROUGE scores:**
- **ROUGE-1**: unigram overlap — captures vocabulary coverage
- **ROUGE-2**: bigram overlap — captures phrase fluency
- **ROUGE-L**: longest common subsequence — captures sentence-level structure

All three scores are 0–1; higher is better. For abstractive summarization, ROUGE-L F1 ≥ 0.3 is generally considered good.

## 1. Imports

In [ ]:
from bellmira.evaluators import ModelSummarizationEvaluator

## 2. Prepare (document, reference_summary) pairs

Each pair is a tuple of `(full_document_text, human_written_reference_summary)`.
The reference is the ground truth the model is scored against.

In [ ]:
PAIRS = [
    (
        """O Millennium BCP é o maior banco privado em Portugal. Foi fundado em 1985 e tem a sua sede em Lisboa.
O banco oferece uma vasta gama de produtos e serviços financeiros, incluindo contas à ordem, crédito habitação,
cartões, seguros, investimentos e soluções digitais como o MBway. Em 2023, o banco registou lucros recordes
e continuou a expandir a sua presença no mercado. O banco tem operações em vários países, incluindo Polónia,
Moçambique e Angola.""",
        "O Millennium BCP é o maior banco privado português, fundado em 1985, com sede em Lisboa e presença internacional."
    ),
    (
        """Retrieval-Augmented Generation (RAG) is a technique that enhances large language models by incorporating
external knowledge at inference time. Instead of relying solely on parameters learned during training,
a RAG system retrieves relevant documents from a knowledge base and provides them as additional context
to the model. This approach significantly reduces hallucinations, improves factual accuracy, and allows
the model to answer questions about information that was not part of its training data.""",
        "RAG improves LLM accuracy by retrieving external documents at inference time, reducing hallucinations."
    ),
    (
        """In the beginning, God created the heavens and the earth. The earth was formless and empty, and darkness
covered the deep waters. And the Spirit of God was hovering over the surface of the waters.
Then God said, 'Let there be light,' and there was light. And God saw that the light was good.
Then he separated the light from the darkness. God called the light 'day' and the darkness 'night.'
And evening passed and morning came, marking the first day.""",
        "God created light and separated it from darkness on the first day of creation."
    ),
    (
        """Transformer models use self-attention mechanisms to process sequences in parallel, unlike recurrent
neural networks which process tokens sequentially. The attention mechanism computes relationships between
all pairs of tokens in a sequence simultaneously. This parallelism enables much faster training on modern
GPUs. The original Transformer architecture was introduced by Vaswani et al. in 2017 and has since become
the dominant paradigm for natural language processing tasks.""",
        "Transformers use parallel self-attention instead of sequential processing, enabling faster training than RNNs."
    ),
]

## 3. Configuration

In [ ]:
MODEL_URL = "https://chatqwen25-15-cc-sum-backend.dat-aip-millm.qa.mbcp.cloud/"

## 4. Initialise the evaluator

In [ ]:
evaluator = ModelSummarizationEvaluator(
    url=MODEL_URL,
    pairs=PAIRS,
    temperature=0.0,
    max_tokens=128,
)

print(f"Model: {evaluator.model_name}")

## 5. Run evaluation

In [ ]:
results = evaluator.evaluate()

for i, r in enumerate(results):
    if r.get("ROUGE1_F1") is None:
        print(f"\n[{i+1}] ERROR: {r.get('Error')}")
        continue
    print(f"\n[{i+1}] R1={r['ROUGE1_F1']:.3f}  R2={r['ROUGE2_F1']:.3f}  RL={r['ROUGEL_F1']:.3f}  "
          f"compression={r['Compression_ratio']:.3f}  latency={r['Latency']:.2f}s")
    print(f"     Reference:  {r['Reference_summary']}")
    print(f"     Generated:  {r['Generated_summary']}")

## 6. Extract aggregate metrics

In [ ]:
metrics = evaluator.extract_threshold_metrics(results)

print(f"Pairs evaluated:      {metrics['Pairs_evaluated']}")
print(f"Avg ROUGE-1 F1:       {metrics['Avg_ROUGE1_F1']}")
print(f"Avg ROUGE-2 F1:       {metrics['Avg_ROUGE2_F1']}")
print(f"Avg ROUGE-L F1:       {metrics['Avg_ROUGEL_F1']}")
print(f"Avg ROUGE-1 Precision:{metrics['Avg_ROUGE1_precision']}")
print(f"Avg ROUGE-1 Recall:   {metrics['Avg_ROUGE1_recall']}")
print(f"Avg compression:      {metrics['Avg_compression_ratio']}  (lower = more concise)")
print(f"Avg latency:          {metrics['Avg_latency']}s")
print(f"Avg prompt tokens:    {metrics['Avg_prompt_tokens']}")
print(f"Avg completion tok:   {metrics['Avg_completion_tokens']}")

## 7. Customise the summarisation prompt

In [ ]:
# Evaluate with a Portuguese-specific prompt
evaluator_pt = ModelSummarizationEvaluator(
    url=MODEL_URL,
    pairs=PAIRS[:2],
    system_prompt="És um assistente de sumarização conciso. Responde apenas com o resumo, sem introduções.",
    user_template="Resume o seguinte texto em duas frases:\n\n{document}",
    temperature=0.0,
    max_tokens=128,
)

res_pt = evaluator_pt.evaluate()
for r in res_pt:
    print(f"R1={r.get('ROUGE1_F1')}  RL={r.get('ROUGEL_F1')}")
    print(f"  {r.get('Generated_summary')}\n")

## 8. Log to MLflow

In [ ]:
import mlflow

mlflow.set_experiment("summarization_evaluation")
with mlflow.start_run(run_name="Qwen25-sum-qa"):
    loggable = {k: v for k, v in metrics.items() if isinstance(v, (int, float)) and v is not None}
    mlflow.log_metrics(loggable)
    mlflow.log_param("model_url", MODEL_URL)
    mlflow.log_param("pairs_count", len(PAIRS))
    print("Logged:", loggable)